# Preliminary Analysis — dunnhumby "Breakfast at the Frat"

**Dataset:** [`dunnhumby - Breakfast at the Frat.xlsx`](https://www.kaggle.com/datasets/frtgnn/dunnhumby-the-complete-journey)
(sales & promotional time series for a sample of stores, 156 weeks, Jan 2009 – Dec 2011).

Four sheets:

| sheet | content |
| --- | --- |
| `Glossary` | variable definitions |
| `dh Store Lookup` | store dimensions (city/state, MSA, segment, size, avg weekly baskets) |
| `dh Products Lookup` | product dimensions (UPC, manufacturer, category, sub-category, size) |
| `dh Transaction Data` | weekly store × product sales and promotion flags |

**Purpose of this notebook:** get familiar with the data before the deeper EDA
(`EDA.ipynb`). Understand its shape, quality, coverage, main dimensions and
first-order patterns:

1. What is in the data — volume, granularity, date range, keys?
2. Is the data clean — missing values, duplicates, referential integrity, anomalies?
3. What do the key metrics (units, spend, price) look like?
4. How does sales evolve over time — trend and seasonality?
5. Which categories / products / manufacturers dominate?
6. Do promotions (feature, display, TPR) show up in sales and price?
7. How do stores compare?

> **Note:** all figures below are exploratory, not causal.

## 1. Setup

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (11, 4.5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.titleweight"] = "bold"

DATA = Path("../data/dunnhumby - Breakfast at the Frat.xlsx")
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pd.set_option("display.max_columns", 50)

print(f"pandas {pd.__version__} | seaborn {sns.__version__}")

## 2. Load the data

Headers sit on row 2 of each sheet (row 1 is a title banner), so `header=1`.
The transaction sheet is large (~525k rows), expect ~30 s to parse.

In [ ]:
xl = pd.ExcelFile(DATA, engine="openpyxl")

glossary = pd.read_excel(xl, "Glossary", header=3)
stores = pd.read_excel(xl, "dh Store Lookup", header=1)
products = pd.read_excel(xl, "dh Products Lookup", header=1)
tx = pd.read_excel(xl, "dh Transaction Data", header=1)

for name, df in {"stores": stores, "products": products, "tx": tx}.items():
    print(f"{name:>10}: {df.shape[0]:>7,} rows x {df.shape[1]} cols")

In [ ]:
#Unnamed column is an excel parsing error. Do not include in 
glossary = glossary.drop(columns=["Unnamed: 3"])

In [ ]:
glossary[glossary["TABLE"].str.contains("store")]

In [ ]:
glossary[glossary["TABLE"].str.contains("products")]

In [ ]:
glossary[glossary["TABLE"].str.contains("data")]

## 3. Data overview

**Transaction grain** 

> **Info**: one row per `WEEK_END_DATE × STORE_NUM × UPC` —
a panel of weekly store-product observations.

In [ ]:
summary = pd.DataFrame({
    "dtype": tx.dtypes.astype(str),
    "non-null": tx.notna().sum(),
    "nulls": tx.isna().sum(),
    "nulls, %": (tx.isna().mean() * 100).round(2),
    "n_unique": tx.nunique(),
})
summary

In [ ]:
print("Date range      :", tx.WEEK_END_DATE.min().date(), "→", tx.WEEK_END_DATE.max().date())
print("Weeks           :", tx.WEEK_END_DATE.nunique())
print("Stores in data  :", tx.STORE_NUM.nunique(),
      f"| in lookup: {stores.STORE_ID.nunique()} unique IDs in {len(stores)} rows")
print("Products in data:", tx.UPC.nunique(), "| in lookup:", products.UPC.nunique())
print("Total units     :", f"{tx.UNITS.sum():,.0f}")
print("Total spend     : $%s" % f"{tx.SPEND.sum():,.0f}")
print("Rows / week     :", f"{len(tx) / tx.WEEK_END_DATE.nunique():,.0f} (WEEK_END_DATE × STORE_NUM × UPC)")

**Store dimension**

In [ ]:
summary_stores = pd.DataFrame({
    "dtype": stores.dtypes.astype(str),
    "non-null": stores.notna().sum(),
    "nulls": stores.isna().sum(),
    "nulls, %": (stores.isna().mean() * 100).round(2),
    "n_unique": stores.nunique(),
})
summary_stores

In [ ]:
print(stores.shape)
stores.head()

**product dimension**

In [ ]:
summary_products = pd.DataFrame({
    "dtype": products.dtypes.astype(str),
    "non-null": products.notna().sum(),
    "nulls": products.isna().sum(),
    "nulls, %": (products.isna().mean() * 100).round(2),
    "n_unique": products.nunique(),
})
summary_products


In [ ]:
print(products.shape)
products.head()

## 4. Data quality

Checks: missing values, duplicate rows, key integrity between facts and
dimensions, and value-level anomalies.

In [ ]:
print("Duplicate rows in tx:", tx.duplicated().sum())

# referential integrity
orphan_stores = (~tx.STORE_NUM.isin(stores.STORE_ID)).sum()
orphan_upcs = (~tx.UPC.isin(products.UPC)).sum()
unused_stores = (~stores.STORE_ID.isin(tx.STORE_NUM)).sum()
unused_upcs = (~products.UPC.isin(tx.UPC)).sum()
print(f"tx rows with store not in lookup : {orphan_stores}")
print(f"tx rows with UPC   not in lookup : {orphan_upcs}")
print(f"lookup stores w/o transactions   : {unused_stores}")
print(f"lookup products w/o transactions : {unused_upcs}")

In [ ]:
# duplicate keys inside the dimension tables?
print("duplicate UPCs in products :", products.duplicated("UPC").sum())
print(f"store lookup: {len(stores)} rows vs {stores.STORE_ID.nunique()} unique STORE_IDs")
dup_stores = stores[stores.duplicated("STORE_ID", keep=False)].sort_values("STORE_ID")
print(f"{dup_stores.STORE_ID.nunique()} store IDs appear twice — identical except SEG_VALUE_NAME:")
dup_stores

In [ ]:
tx_anomalies = {
    "UNITS <= 0": int((tx.UNITS <= 0).sum()),
    "SPEND < 0": int((tx.SPEND < 0).sum()),
    "SPEND == 0": int((tx.SPEND == 0).sum()),
    "PRICE <= 0": int((tx.PRICE <= 0).sum()),
    "BASE_PRICE null": int(tx.BASE_PRICE.isna().sum()),
    "PRICE null": int(tx.PRICE.isna().sum()),
    "VISITS == 0": int((tx.VISITS == 0).sum()),
    "HHS > VISITS": int((tx.HHS > tx.VISITS).sum()),
    "FEATURE & DISPLAY & TPR all 1": int(((tx.FEATURE == 1) & (tx.DISPLAY == 1) & (tx.TPR_ONLY == 1)).sum()),
}
pd.Series(tx_anomalies, name="rows")

In [ ]:
# rows with missing or non-positive prices: are they tiny?
bad = tx[tx.PRICE.isna() | tx.BASE_PRICE.isna() | (tx.PRICE <= 0)]
print(f"rows with price issues: {len(bad)} ({len(bad)/len(tx):.3%} of data)")
bad[["WEEK_END_DATE", "STORE_NUM", "UPC", "UNITS", "SPEND", "PRICE", "BASE_PRICE"]].head(10)

In [ ]:
# coverage: how many weeks does each store contribute?
weeks_per_store = tx.groupby("STORE_NUM").WEEK_END_DATE.nunique()
print(weeks_per_store.describe().to_string())
print("\nstores with < 156 weeks:", (weeks_per_store < 156).sum())

# expected balanced panel size vs actual
expected = tx.STORE_NUM.nunique() * tx.UPC.nunique() * tx.WEEK_END_DATE.nunique()
print(f"\nactual rows {len(tx):,} vs fully-balanced panel {expected:,} "
      f"({len(tx)/expected:.1%} — missing store-weeks are normal: shelf gaps/out-of-stock)")

### Data quality — findings

No hard blockers: no duplicate rows in the transaction table, full referential
integrity of facts to dimensions. Issues found and handled below:

- `PRICE`/`BASE_PRICE` missing or non-positive on 209 rows (0.04%) — ignored where relevant.
- Store lookup has **79 rows for 77 stores** (2 IDs duplicated with conflicting `SEG_VALUE_NAME`) — deduplicated before store-level joins.
- 3 of 58 products never appear in transactions; 1 store covers only 131 of 156 weeks.
- `PARKING_SPACE_QTY` is missing for 52/79 rows — unusable as a feature.
- Panel is 79.5% of a fully balanced grid — missing store-weeks are normal (shelf gaps / out-of-stock).